In [12]:
from pathlib import Path
import pandas as pd

In [13]:
# Paths
label_path = '/home/ec2-user/Jul2025/labels/' # label folder with all the labels

# Orre et al. 3365 markers
orre_marker_path = Path(label_path+'markers.txt') # path to the orre marker file
orre_mcluster_path = Path(label_path+'markers_mclusters.txt') # path to the orre mcluster file

# Uniprot Go markers
uniprot_marker_path = Path(label_path+'uniprot_go_markers_grouped.txt') # path to the uniprot go marker file

In [21]:
# Load reference label tables
ld_path = Path(label_path) / 'uniprot_go_markers_grouped.txt'
LD = (
    pd.read_csv(ld_path, sep='\t')
      .rename(columns={'Protein': 'Protein_ID'})
      .set_index('Protein_ID')
)

marker_path = Path(label_path) / 'markers.txt'
LD_marker = (
    pd.read_csv(marker_path, sep='\t')
      .rename(columns={'Protein': 'Protein_ID'})
      .set_index('Protein_ID')
)

# Reconcile marker localizations with LD where they disagree
LD_marker_new = LD_marker.copy()
shared_proteins = LD_marker_new.index.intersection(LD.index)

mismatch_mask = LD_marker_new.loc[shared_proteins, 'Localization'] != LD.loc[shared_proteins, 'Localization']
proteins_updated = shared_proteins[mismatch_mask]

LD_marker_new.loc[proteins_updated, 'Localization'] = LD.loc[proteins_updated, 'Localization']
LD_marker_new_path = Path(label_path) / 'markers_modified_1.txt'
LD_marker_new.to_csv(LD_marker_new_path, sep='\t', index=True)

print(f"Updated localizations for {len(proteins_updated)} proteins.")
print(f"Revised marker file written to {LD_marker_new_path}.")

if len(proteins_updated):
    change_summary = (
        pd.DataFrame({
            'Original_Localization': LD_marker.loc[proteins_updated, 'Localization'],
            'Revised_Localization': LD_marker_new.loc[proteins_updated, 'Localization']
        })
        .value_counts()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )

    display(change_summary)
else:
    print('No localization discrepancies found between LD_marker and LD.')

Updated localizations for 250 proteins.
Revised marker file written to /home/ec2-user/Jul2025/labels/markers_modified_1.txt.


,Original_Localization,Revised_Localization,count
0,Secretory,Mitochondria,66
1,Cytosol,Nucleus,42
2,Nucleus,Secretory,40
3,Cytosol,Secretory,22
4,Secretory,Nucleus,19
5,Secretory,Cytosol,16
6,Nucleus,Cytosol,11
7,Cytosol,Mitochondria,11
8,Nucleus,Mitochondria,8
9,Mitochondria,Cytosol,5


In [18]:
# Cross-check cluster proteins against LD_marker_new localizations
clusters_to_check = ['mclust3', 'mclust5']
cluster_localization_checks = []

orre_mclusters = pd.read_csv(orre_mcluster_path, sep='\t')

for cluster_id in clusters_to_check:
    cluster_df = orre_mclusters[orre_mclusters['Cluster'] == cluster_id]
    proteins = cluster_df['Protein'].tolist()
    print(f"\n=== {cluster_id} in LD_marker_new ===")
    if not proteins:
        print("No proteins found in this cluster.")
        continue

    # Fetch localizations from the updated marker table
    localizations = LD_marker_new.loc[LD_marker_new.index.intersection(proteins), ['Localization']]
    merged = cluster_df.set_index('Protein').join(localizations, rsuffix='_LD_marker_new')

    display(merged.head())

    # Summaries
    summary_counts = merged['Localization'].value_counts(dropna=False).to_dict()
    cluster_localization_checks.append({
        'mcluster': cluster_id,
        'n_cluster_proteins': len(cluster_df),
        'n_found_in_LD_marker_new': localizations.shape[0],
        'localization_counts': summary_counts
    })

if cluster_localization_checks:
    summary_df = pd.DataFrame(cluster_localization_checks)
    display(summary_df)


=== mclust3 in LD_marker_new ===


,Cluster,Localization
Protein,,
ABCD1,mclust3,Secretory
ABHD16A,mclust3,Secretory
ACSL1,mclust3,Secretory
ACSL4,mclust3,Secretory
AGRN,mclust3,Secretory



=== mclust5 in LD_marker_new ===


,Cluster,Localization
Protein,,
ADARB1,mclust5,Nucleus
AGO2,mclust5,Nucleus
AHDC1,mclust5,Nucleus
AZI1,mclust5,Nucleus
BARD1,mclust5,Nucleus


,mcluster,n_cluster_proteins,n_found_in_LD_marker_new,localization_counts
0,mclust3,252,252,"{'Secretory': 189, 'Mitochondria': 55, 'Cytoso..."
1,mclust5,192,192,"{'Nucleus': 156, 'Secretory': 31, 'Cytosol': 5}"


In [19]:
# Create a modified copy with cluster-based localization adjustments
LD_marker_modified = LD_marker_new.copy()

# For mclust3: replace non-Mitochondria localization with Mitochondria
mclust3_proteins = orre_mclusters.loc[orre_mclusters['Cluster'] == 'mclust3', 'Protein']
idx3 = LD_marker_modified.index.intersection(mclust3_proteins)
mask3 = LD_marker_modified.loc[idx3, 'Localization'] != 'Mitochondria'
LD_marker_modified.loc[idx3[mask3], 'Localization'] = 'Mitochondria'

# For mclust5: replace non-Secretory localization with Secretory
mclust5_proteins = orre_mclusters.loc[orre_mclusters['Cluster'] == 'mclust5', 'Protein']
idx5 = LD_marker_modified.index.intersection(mclust5_proteins)
mask5 = LD_marker_modified.loc[idx5, 'Localization'] != 'Secretory'
LD_marker_modified.loc[idx5[mask5], 'Localization'] = 'Secretory'

# Save the modified table
output_path = label_path + 'markers_modified_2.txt'
LD_marker_modified.to_csv(path_or_buf=output_path, sep='\t', index=True)
print(f"Modified markers saved to: {output_path}")

# Summaries similar to before
summary_rows = []
for cluster_id, proteins in [('mclust3', idx3), ('mclust5', idx5)]:
    loc_counts = LD_marker_modified.loc[proteins, 'Localization'].value_counts(dropna=False).to_dict()
    summary_rows.append({
        'mcluster': cluster_id,
        'n_cluster_proteins': len(proteins),
        'localization_counts': loc_counts
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

Modified markers saved to: /home/ec2-user/Jul2025/labels/markers_modified_2.txt


,mcluster,n_cluster_proteins,localization_counts
0,mclust3,252,{'Mitochondria': 252}
1,mclust5,192,{'Secretory': 192}


In [20]:
# Compare original LD_marker vs LD_marker_modified to quantify discrepancies
comparison_df = (
    LD_marker[['Localization']]
    .rename(columns={'Localization': 'Localization_original'})
    .join(LD_marker_modified[['Localization']].rename(columns={'Localization': 'Localization_modified'}))
)
comparison_df['changed'] = comparison_df['Localization_original'] != comparison_df['Localization_modified']

changed_entries = comparison_df[comparison_df['changed']]
total_changed = len(changed_entries)
print(f"Total entries changed: {total_changed}")

if total_changed > 0:
    # Detailed change summary
    change_summary = (
        changed_entries
        .groupby(['Localization_original', 'Localization_modified'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )

    # Add per-cluster counts for additional insight
    cluster_change_summary = (
        orre_mclusters.set_index('Protein')
        .join(changed_entries[['Localization_original', 'Localization_modified']])
        .dropna(subset=['Localization_original'])
        .groupby('Cluster')
        .size()
        .reset_index(name='n_changed')
        .sort_values('n_changed', ascending=False)
    )

    display(change_summary)
    display(cluster_change_summary)

    # Show sample of changed entries with cluster info
    merged_preview = (
        changed_entries
        .join(orre_mclusters.set_index('Protein')['Cluster'])
        .sort_values(['Cluster', 'Localization_original'])
    )
    display(merged_preview.head(10))
else:
    print("No localization changes detected between the original and modified tables.")

Total entries changed: 595


,Localization_original,Localization_modified,changed
Protein,,,
AAAS,Secretory,Nucleus,True
ABCD1,Secretory,Mitochondria,True
ABHD16A,Secretory,Mitochondria,True
ABHD6,Secretory,Mitochondria,True
ACAP2,Cytosol,Secretory,True


,Localization_original,Localization_modified,count
0,Cytosol,Mitochondria,11
1,Cytosol,Nucleus,42
2,Cytosol,Secretory,22
3,Mitochondria,Cytosol,5
4,Mitochondria,Nucleus,5
5,Mitochondria,Secretory,5
6,Nucleus,Cytosol,6
7,Nucleus,Mitochondria,8
8,Nucleus,Secretory,201
9,Secretory,Cytosol,11
